In [0]:
# Databricks notebook source
# File: silver_to_gold/02_silver_to_gold.py
# Project: MIGRATION-DBX-001
# Purpose: Read Silver Delta → build star schema Gold layer
# Write order: Dimensions → Fact → Aggregates → OPTIMIZE
# =============================================================================

# COMMAND ----------
# MAGIC %md
# MAGIC ## Silver → Gold
# MAGIC **Layers built:**
# MAGIC - `dim_customers`, `dim_products`, `dim_sellers`, `dim_date` (dimensions)
# MAGIC - `fact_orders` (order-grain fact with payment + review enrichment)
# MAGIC - `agg_daily_revenue`, `agg_seller_performance` (aggregates)
# MAGIC
# MAGIC **Run after:** `01_bronze_to_silver` completes successfully.

In [0]:
%run ../config/00_shared_config

In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, DecimalType, DoubleType,
    StringType, DateType, TimestampType, BooleanType
)
from delta.tables import DeltaTable

log("Silver → Gold notebook starting.")

In [0]:
# COMMAND ----------
# =============================================================================
# HELPER: MERGE into Gold
# Simpler than Silver MERGE — Gold tables are clean, PKs are well-defined
# =============================================================================

def merge_into_gold(table_uc: str, table_path: str, df, merge_cols: list, overwrite: bool = False):
    """
    Writes df to Gold Delta table.
    overwrite=True: used for aggregates (full recalculation each run).
    overwrite=False: MERGE on merge_cols (used for dims and facts).
    """
    table_exists = spark.catalog.tableExists(table_uc)

    if overwrite or not table_exists:
        write_mode = "overwrite"
        (df.write
             .format("delta")
             .mode(write_mode)
             .option("overwriteSchema", "true")
             .option("path", table_path)
             .saveAsTable(table_uc))
        log(f"  Wrote {df.count():,} rows to {table_uc} (overwrite)")
        return

    # MERGE path
    delta_table = DeltaTable.forName(spark, table_uc)
    merge_cond  = " AND ".join([f"t.{c} = s.{c}" for c in merge_cols])
    update_cols = [c for c in df.columns if c not in merge_cols]
    update_map  = {c: f"s.{c}" for c in update_cols}

    (delta_table.alias("t")
        .merge(df.alias("s"), merge_cond)
        .whenMatchedUpdate(set=update_map)
        .whenNotMatchedInsertAll()
        .execute())
    log(f"  MERGE complete: {table_uc} ({df.count():,} rows)")

In [0]:
log("Reading Silver tables...")

sv_orders    = spark.table(f"{UC_SILVER}.orders")
sv_items     = spark.table(f"{UC_SILVER}.order_items")
sv_payments  = spark.table(f"{UC_SILVER}.order_payments")
sv_reviews   = spark.table(f"{UC_SILVER}.order_reviews")
sv_customers = spark.table(f"{UC_SILVER}.customers").cache()
sv_products  = spark.table(f"{UC_SILVER}.products").cache()
sv_sellers   = spark.table(f"{UC_SILVER}.sellers").cache()
sv_geo       = spark.table(f"{UC_SILVER}.geolocation").cache()   # already deduped in Silver
sv_cat       = spark.table(f"{UC_SILVER}.category_translation").cache()

log("Silver tables loaded.")

In [0]:
# COMMAND ----------
# =============================================================================
# DIMENSION 1: dim_customers
# SCD Type 1 — just keep current state of each customer
# Enrich with geolocation (lat/lng) via zip code join
# =============================================================================
log("Building dim_customers...")

dim_customers = (
    sv_customers
    .join(sv_geo, on="customer_zip_code_prefix", how="left")
    .select(
        # Surrogate key — MD5 hash of natural key (deterministic, no sequence needed)
        F.md5(F.col("customer_id")).alias("customer_sk"),
        F.col("customer_id"),
        F.col("customer_unique_id"),
        F.col("customer_zip_code_prefix"),
        F.col("customer_city"),
        F.col("customer_state"),
        F.col("geolocation_lat"),
        F.col("geolocation_lng"),
        F.current_timestamp().alias("gold_loaded_at"),
    )
)

merge_into_gold(
    table_uc   = f"{UC_GOLD}.dim_customers",
    table_path = f"{GOLD_BASE}/dim_customers",
    df         = dim_customers,
    merge_cols = ["customer_id"],
)

In [0]:
# COMMAND ----------
# =============================================================================
# DIMENSION 2: dim_products
# Enrich with English category name via LEFT JOIN on category_translation
# ~600 products have NULL category → they get 'unknown' in English
# =============================================================================
log("Building dim_products...")

dim_products = (
    sv_products
    .join(sv_cat, sv_products.product_category_name_pt == sv_cat.product_category_name_pt, how="left")
    .select(
        F.md5(sv_products.product_id).alias("product_sk"),
        sv_products.product_id,
        sv_products.product_category_name_pt,
        # NULL category → 'unknown'
        F.coalesce(sv_cat.product_category_name_en, F.lit("unknown")).alias("product_category_name_en"),
        sv_products.product_weight_g,
        sv_products.product_photos_qty,
        sv_products.product_length_cm,
        sv_products.product_height_cm,
        sv_products.product_width_cm,
        F.current_timestamp().alias("gold_loaded_at"),
    )
)

merge_into_gold(
    table_uc   = f"{UC_GOLD}.dim_products",
    table_path = f"{GOLD_BASE}/dim_products",
    df         = dim_products,
    merge_cols = ["product_id"],
)

In [0]:
# COMMAND ----------
# =============================================================================
# DIMENSION 3: dim_sellers
# Enrich with geolocation
# =============================================================================
log("Building dim_sellers...")

dim_sellers = (
    sv_sellers
    .join(sv_geo, on=sv_sellers.seller_zip_code_prefix == sv_geo.geolocation_zip_code_prefix, how="left")
    .select(
        F.md5(sv_sellers.seller_id).alias("seller_sk"),
        sv_sellers.seller_id,
        sv_sellers.seller_zip_code_prefix,
        sv_sellers.seller_city,
        sv_sellers.seller_state,
        sv_geo.geolocation_lat,
        sv_geo.geolocation_lng,
        F.current_timestamp().alias("gold_loaded_at"),
    )
)

merge_into_gold(
    table_uc   = f"{UC_GOLD}.dim_sellers",
    table_path = f"{GOLD_BASE}/dim_sellers",
    df         = dim_sellers,
    merge_cols = ["seller_id"],
)

In [0]:
# COMMAND ----------
# =============================================================================
# DIMENSION 4: dim_date
# Calendar spine generated once covering Olist data range (2016–2020)
# Generated programmatically — no source table needed
# =============================================================================
log("Building dim_date...")

# Generate date range as a DataFrame
date_spine = (
    spark.sql("""
        SELECT explode(sequence(
            to_date('2016-01-01'),
            to_date('2020-12-31'),
            interval 1 day
        )) AS full_date
    """)
)

dim_date = date_spine.select(
    F.date_format(F.col("full_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
    F.col("full_date"),
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.weekofyear("full_date").alias("week_of_year"),
    F.dayofweek("full_date").alias("day_of_week"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.when(F.dayofweek("full_date").isin(1, 7), F.lit(True))
     .otherwise(F.lit(False)).alias("is_weekend"),
    F.current_timestamp().alias("gold_loaded_at"),
)

merge_into_gold(
    table_uc   = f"{UC_GOLD}.dim_date",
    table_path = f"{GOLD_BASE}/dim_date",
    df         = dim_date,
    merge_cols = ["date_key"],
)

In [0]:
# COMMAND ----------
# =============================================================================
# PRE-AGGREGATE: payments and items before joining to fact
# CRITICAL: without this, JOIN causes row explosion (PROB-06 learning moment)
#
# Example of PROB-06 if you don't do this:
#   order ABC has 3 items and 2 payments
#   Direct JOIN: 3 * 2 = 6 rows per order in result → revenue 2x inflated
#   Fix: aggregate both sides to order grain FIRST, then join
# =============================================================================
log("Pre-aggregating items and payments to order grain...")

# Items aggregated to order grain
items_agg = (
    sv_items
    .groupBy("order_id")
    .agg(
        F.sum("price").cast(DecimalType(12,2)).alias("total_item_value"),
        F.sum("freight_value").cast(DecimalType(12,2)).alias("total_freight_value"),
        F.count("order_item_id").alias("item_count"),
    )
)

# Payments aggregated to order grain
payments_agg = (
    sv_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").cast(DecimalType(12,2)).alias("total_payment_value"),
        F.first("payment_type").alias("primary_payment_type"),
        F.max("payment_installments").alias("max_installments"),
        F.countDistinct("payment_type").alias("payment_type_count"),
    )
)

# Reviews aggregated to order grain (some orders have multiple reviews)
reviews_agg = (
    sv_reviews
    .filter(F.col("review_id").isNotNull())   # exclude PROB-03 quarantined NULLs
    .groupBy("order_id")
    .agg(
        F.avg("review_score").cast(DecimalType(3,2)).alias("avg_review_score"),
    )
)

log("Pre-aggregation complete.")

In [0]:
# COMMAND ----------
# =============================================================================
# FACT: fact_orders
# Order-grain fact table
# Joins: orders + items_agg + payments_agg + reviews_agg + dim_customers + dim_date
# =============================================================================
log("Building fact_orders...")

fact_orders = (
    sv_orders
    # Join pre-aggregated sides (no row explosion risk)
    .join(items_agg,    on="order_id", how="left")
    .join(payments_agg, on="order_id", how="left")
    .join(reviews_agg,  on="order_id", how="left")
    # Join dim_customers to get SK
    .join(
        dim_customers.select("customer_id", "customer_sk"),
        on="customer_id", how="left"
    )
    .select(
        # Surrogate key for fact
        F.md5(sv_orders.order_id).alias("order_sk"),
        sv_orders.order_id,
        F.col("customer_sk"),
        # Date key — join to dim_date in queries (not pre-joined here for flexibility)
        F.date_format("order_purchase_ts", "yyyyMMdd")
         .cast(IntegerType()).alias("date_key"),
        sv_orders.order_status,
        sv_orders.order_purchase_ts,
        sv_orders.order_delivered_customer_ts,
        # Derived: delivery days (NULL if not delivered)
        F.when(
            sv_orders.order_delivered_customer_ts.isNotNull(),
            F.datediff(
                sv_orders.order_delivered_customer_ts,
                sv_orders.order_purchase_ts
            )
        ).alias("delivery_days"),
        # Item metrics
        F.coalesce(F.col("total_item_value"),    F.lit(0)).cast(DecimalType(12,2)).alias("total_item_value"),
        F.coalesce(F.col("total_freight_value"),  F.lit(0)).cast(DecimalType(12,2)).alias("total_freight_value"),
        F.coalesce(F.col("item_count"),           F.lit(0)).alias("item_count"),
        # Payment metrics
        F.coalesce(F.col("total_payment_value"),  F.lit(0)).cast(DecimalType(12,2)).alias("total_payment_value"),
        F.col("primary_payment_type"),
        F.col("max_installments"),
        # Review
        F.col("avg_review_score"),
        F.current_timestamp().alias("gold_loaded_at"),
    )
)

merge_into_gold(
    table_uc   = f"{UC_GOLD}.fact_orders",
    table_path = f"{GOLD_BASE}/fact_orders",
    df         = fact_orders,
    merge_cols = ["order_id"],
)


In [0]:
# COMMAND ----------
# =============================================================================
# AGGREGATE 1: agg_daily_revenue
# Overwrite each run — aggregate is derived, not append-friendly
# =============================================================================
log("Building agg_daily_revenue...")

agg_daily_revenue = (
    spark.table(f"{UC_GOLD}.fact_orders")
    .withColumn("order_date", F.to_date("order_purchase_ts"))
    .groupBy("order_date")
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("total_payment_value").cast(DecimalType(14,2)).alias("total_revenue"),
        F.avg("total_payment_value").cast(DecimalType(10,2)).alias("avg_order_value"),
        F.sum("item_count").alias("total_items_sold"),
        F.avg(
            F.when(F.col("delivery_days").isNotNull(), F.col("delivery_days"))
        ).cast(DecimalType(6,2)).alias("avg_delivery_days"),
        F.current_timestamp().alias("gold_loaded_at"),
    )
    .orderBy("order_date")
)

merge_into_gold(
    table_uc   = f"{UC_GOLD}.agg_daily_revenue",
    table_path = f"{GOLD_BASE}/agg_daily_revenue",
    df         = agg_daily_revenue,
    merge_cols = ["order_date"],
    overwrite  = True,   # full recalc
)


In [0]:
# COMMAND ----------
# =============================================================================
# AGGREGATE 2: agg_seller_performance
# Seller-grain aggregate — revenue, review scores, delivery performance
# =============================================================================
log("Building agg_seller_performance...")

# Join items back to get seller_id (fact_orders doesn't have seller_id directly)
seller_perf_base = (
    spark.table(f"{UC_GOLD}.fact_orders")
    .join(sv_items.select("order_id", "seller_id", "price").distinct(),
          on="order_id", how="left")
    .join(dim_sellers.select("seller_id", "seller_sk", "seller_state"),
          on="seller_id", how="left")
)

agg_seller_performance = (
    seller_perf_base
    .groupBy("seller_id", "seller_sk", "seller_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("price").cast(DecimalType(14,2)).alias("total_revenue"),
        F.avg("avg_review_score").cast(DecimalType(4,2)).alias("avg_review_score"),
        F.avg(
            F.when(F.col("delivery_days").isNotNull(), F.col("delivery_days"))
        ).cast(DecimalType(6,2)).alias("avg_delivery_days"),
        F.current_timestamp().alias("gold_loaded_at"),
    )
)

merge_into_gold(
    table_uc   = f"{UC_GOLD}.agg_seller_performance",
    table_path = f"{GOLD_BASE}/agg_seller_performance",
    df         = agg_seller_performance,
    merge_cols = ["seller_id"],
    overwrite  = True,
)

In [0]:
# COMMAND ----------
# =============================================================================
# CELL: OPTIMIZE + ZORDER
# Run AFTER all Gold writes complete
# Why: OPTIMIZE consolidates small files written by Spark into larger files
#      ZORDER co-locates related data physically → faster range queries
# Only high-query tables need this — fact and agg tables
# =============================================================================
log("Running OPTIMIZE + ZORDER on Gold tables...")

# fact_orders: most queried by date and customer → ZORDER on these
spark.sql(f"""
    OPTIMIZE {UC_GOLD}.fact_orders
    ZORDER BY (order_purchase_ts, customer_sk)
""")
log("  OPTIMIZE + ZORDER complete: fact_orders")

# agg_daily_revenue: always queried by date
spark.sql(f"""
    OPTIMIZE {UC_GOLD}.agg_daily_revenue
    ZORDER BY (order_date)
""")
log("  OPTIMIZE + ZORDER complete: agg_daily_revenue")

In [0]:
# COMMAND ----------
# =============================================================================
# CELL: Final verification queries
# Run these to confirm Gold layer is correct
# =============================================================================
log("Running verification queries...")

# Row counts across all Gold tables
gold_tables = [
    "dim_customers", "dim_products", "dim_sellers", "dim_date",
    "fact_orders", "agg_daily_revenue", "agg_seller_performance"
]

print("\n" + "="*55)
print("GOLD LAYER — ROW COUNTS")
print("="*55)
for tbl in gold_tables:
    try:
        count = spark.table(f"{UC_GOLD}.{tbl}").count()
        print(f"  {tbl:<30} {count:>10,}")
    except Exception as e:
        print(f"  {tbl:<30}  ERROR: {e}")
print("="*55)

# Sanity: fact_orders row count should match silver.orders
silver_order_count = spark.table(f"{UC_SILVER}.orders").count()
gold_fact_count    = spark.table(f"{UC_GOLD}.fact_orders").count()
print(f"\nSanity check:")
print(f"  silver.orders row count:    {silver_order_count:,}")
print(f"  gold.fact_orders row count: {gold_fact_count:,}")
print(f"  Match: {silver_order_count == gold_fact_count}")

# Top 5 revenue days
print("\nTop 5 Revenue Days:")
(spark.table(f"{UC_GOLD}.agg_daily_revenue")
    .orderBy(F.desc("total_revenue"))
    .select("order_date", "order_count", "total_revenue", "avg_order_value")
    .show(5, truncate=False))

log("Silver → Gold complete.")
